# 第13回：小さなBoltzmann generator

演習20分・採点なし。CPUで実行します。**タンパク質予測ではなく、1次元模型で原理を見る演習**です。第10・12回と同じ $U(x)=(x^2-1)^2+0.25x$、$k_BT=0.25$。今回は区間 $[-2,2]$ に制限した目標分布を使い、理論値も同じ区間で正規化します。

In [ ]:
%pip -q install numpy matplotlib torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json, time
OUT = Path("results"); OUT.mkdir(exist_ok=True)
def U(x): return (x*x-1)**2 + 0.25*x
import torch
torch.set_num_threads(2)
torch.manual_seed(13)


## 1. 単調な折れ線で可逆変換を作る

$x$ の区間を80個に等分し、各区間の確率 $p_i$ をsoftmaxで正に保ちます。一様乱数 $z∼ U(0,1)$ を、累積確率 $c_i$ と区間端 $x_i$ を結ぶ折れ線へ通します。

$c_i≤ z<c_{i+1}$ なら $x=x_i+(z-c_i)Δx/p_i$。傾き $f′=Δx/p_i>0$ なので可逆で、生成密度は $q(x)=p_i/Δx$。区間端で微分は未定義ですが、連続乱数が端に一致する確率は0です。

In [ ]:
N_BINS=80; TAU=0.25
edges=torch.linspace(-2,2,N_BINS+1,dtype=torch.float64);dx=edges[1]-edges[0]
logits=torch.zeros(N_BINS,dtype=torch.float64,requires_grad=True)
# 各区間の平均エネルギーを多項式の積分から厳密に計算する。
def primitive(x): return x**5/5-2*x**3/3+x+0.125*x**2
mean_energy=(primitive(edges[1:])-primitive(edges[:-1]))/dx
initial_p=torch.softmax(logits,0).detach().numpy().copy()
optimizer=torch.optim.Adam([logits],lr=0.04)
loss_history=[]
for step in range(1800):
    logp=torch.log_softmax(logits,0);p=logp.exp()
    loss=(p*(mean_energy/TAU+logp-torch.log(dx))).sum()
    optimizer.zero_grad();loss.backward();optimizer.step()
    loss_history.append(float(loss.detach()))
p=torch.softmax(logits,0).detach().numpy();edge=edges.numpy();delta=float(dx)
print('確率の総和:',p.sum(),'最小確率:',p.min())
assert np.all(p>0) and abs(p.sum()-1)<1e-10

## 2. 損失の意味を確認する

学習したのは逆向きKLの定数項を除いた量です：$L=∑_i p_i[̅U_i/(k_BT)+⁡\log(p_i/Δx)]$。前半は低エネルギーを選び、後半は過度の集中を抑えます。**既知のエネルギーだけで学習し、目標分布の標本を教師にしていません**。有限区間・有限個の折れ線区間が近似です。

In [ ]:
rng=np.random.default_rng(13)
z=rng.random(40000)
cdf=np.r_[0,np.cumsum(p)];cdf[-1]=1
bins=np.clip(np.searchsorted(cdf,z,side='right')-1,0,N_BINS-1)
samples=edge[bins]+(z-cdf[bins])*delta/p[bins]
q=p[bins]/delta
grid=np.linspace(-2,2,20001);reference=np.exp(-U(grid)/TAU);reference/=np.trapezoid(reference,grid)
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].plot(loss_history);axes[0].set(xlabel='training step',ylabel='loss (up to constant)')
axes[1].plot(cdf,edge);axes[1].set(xlabel='uniform latent z',ylabel='generated x')
axes[2].stairs(initial_p/delta,edge,label='before');axes[2].stairs(p/delta,edge,label='after');axes[2].plot(grid,reference,label='target');axes[2].legend();axes[2].set(xlabel='x',ylabel='density')
fig.tight_layout();fig.savefig(OUT/'flow_training.png',dpi=160);plt.show()

## 3. 重み補正で近似を確認する

生成密度が計算できるので、$w=\exp[-U(x)/(k_BT)]/q(x)$ を使います。右の谷の割合を補正前・補正後・理論で比べます。有効標本数 $N_{eff}=(∑w)^2/∑w^2$ は重みの偏りの診断で、未知の状態を見逃していない保証ではありません。

In [ ]:
logw=-U(samples)/TAU-np.log(q);weights=np.exp(logw-logw.max());weights/=weights.sum()
right_raw=float(np.mean(samples>0));right_weighted=float(np.sum(weights*(samples>0)))
right_theory=float(np.trapezoid(reference[grid>0],grid[grid>0]))
ess=float(1/np.sum(weights**2))
print('右の谷：補正前 / 補正後 / 理論',right_raw,right_weighted,right_theory)
print('重みの有効標本数 / 生成数',ess,len(samples))
# 同じ有限区間を使う通常のMetropolis法とも比較する。
x=-1.;mc=[]
for k in range(60000):
    y=x+rng.normal(0,.15)
    if -2<=y<=2 and np.log(rng.random())<min(0,-(U(y)-U(x))/TAU): x=y
    if k>=10000:mc.append(x)
fig,ax=plt.subplots(figsize=(9,4));ax.hist(mc,bins=70,density=True,alpha=.35,label='MCMC');ax.hist(samples,bins=70,density=True,weights=weights,alpha=.35,label='weighted flow');ax.plot(grid,reference,'k',label='target');ax.legend();ax.set(xlabel='x',ylabel='density');fig.tight_layout();fig.savefig(OUT/'comparison.png',dpi=160);plt.show()
report={'model':'80-bin monotone piecewise-linear quantile flow','domain':[-2,2],'tau':TAU,'training_steps':1800,'right_raw':right_raw,'right_reweighted':right_weighted,'right_theory':right_theory,'right_mcmc':float(np.mean(np.array(mc)>0)),'ess':ess,'samples':len(samples)}
(OUT/'metrics.json').write_text(json.dumps(report,indent=2));np.savez(OUT/'flow.npz',edges=edge,probabilities=p,samples=samples,weights=weights)
import shutil
shutil.make_archive('bioinfo13_results','zip',OUT)
print('bioinfo13_results.zip を保存しました')

## 4. 表現の細かさを変える

N_BINSを20、80、160と変えて学習セルから再実行し、生成密度の段差と補正の大きさを見ます。生成した標本の順番を線でつないでも、分子の運動や遷移速度は得られません。

この教材はBoltzmann generatorの可逆変換・エネルギー学習・再重み付けを示す簡略模型です。原著のタンパク質用ネットワークの再現ではありません。Noé et al., Science (2019), DOI:10.1126/science.aaw1147。